# Lecture 5.10 — Guardrail Tripwires and Execution Modes (Blocking vs Parallel)

**Section 05 — Multi-Agent Orchestration & Guardrails**

You already know how to write both kinds of guardrail. This notebook is about what the SDK
is actually doing underneath when a tripwire fires, what each execution mode costs you in
real tokens, and how input guardrails, output guardrails and handoffs compose into one
complete system.

By the end of this notebook you will be able to:

| Goal | Where |
|---|---|
| Read the run loop code that partitions guardrails into two groups | Cell 5 |
| Pull guardrail decisions off a run that **succeeded**, not just one that tripped | Cell 6 |
| Put a real token number on the parallel vs blocking trade-off | Cell 7 |
| See sibling guardrails get cancelled the moment one tripwire fires | Cell 8 |
| Build a streaming check that can stop a long response part way through | Cell 9 |
| Wire input guardrails, output guardrails and a handoff into one flow | Cell 10 |

This notebook does not re-teach how to construct a guardrail. If you need the
`@input_guardrail` and `@output_guardrail` basics, those were covered earlier in this
section.

## Cell 1: Install the OpenAI Agents SDK

This cell installs the `openai-agents` package into the current Colab runtime. The version
is pinned so every example in this notebook behaves exactly as recorded.

If the package is already present in this session, pip will detect that and finish almost
instantly. The `-q` flag keeps the install output short.

If you would rather track the newest release, remove the pin and run
`pip install openai-agents`. You can also substitute any version you prefer.

In [10]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

## Cell 2: API Key Setup with Colab Secrets

The SDK reads your OpenAI key from the `OPENAI_API_KEY` environment variable. In Colab the
safe way to supply it is the built-in Secrets manager, which keeps the key out of the
notebook body entirely.

**How to add the secret in Colab:**

1. Click the **key icon** (🔑) in the left sidebar of Colab.
2. Click **+ Add new secret**.
3. Set the **Name** to `OPENAI_API_KEY`.
4. Paste your key into the **Value** field.
5. Turn on the **Notebook access** toggle for this notebook.
6. Run the cell below.

The cell reads the secret and writes it into `os.environ` so every `Runner.run()` call in
this notebook picks it up automatically.

**Running locally instead of Colab?** Skip the Colab Secrets step and set the variable in
your terminal before launching Jupyter:
`export OPENAI_API_KEY="sk-..."` on macOS or Linux, or
`setx OPENAI_API_KEY "sk-..."` on Windows.

In [11]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

API key loaded: True


## Cell 3: Declare the Model Name

Every agent in this notebook reads its model from a single `MODEL_NAME` variable rather
than hardcoding a model string in each `Agent(...)` definition. Change this one line and
every agent in the notebook switches with it.

That matters more than usual here. Cell 7 measures token consumption, so if you swap the
model you will see different numbers while the pattern stays the same.

In [12]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

print("Using model:", MODEL_NAME)

Using model: gpt-5.4-mini


## Cell 4: Imports

Here is what each import is doing in this notebook.

| Import | Why it is here |
|---|---|
| `asyncio` | `sleep()` for the timing demos, and `create_task()` for the streaming check in Cell 9 |
| `time` | Wall clock measurement to prove sibling cancellation in Cell 8 |
| `BaseModel`, `Field` | Structured output types for the checker agents. `Field` adds descriptions the checker model can read |
| `ResponseTextDeltaEvent` | Raw text delta events from the streaming API. This is the same event type used earlier in the course for token by token streaming |
| `Reasoning` | Sets `reasoning.effort` inside `ModelSettings`. Imported from `openai.types.shared`, not from `agents` |
| `Agent` | The agent class |
| `GuardrailFunctionOutput` | The return type every guardrail function must produce |
| `InputGuardrailTripwireTriggered` | Raised when an input guardrail trips. Carries `guardrail_result` and `run_data` |
| `ModelSettings` | Wraps reasoning effort and verbosity |
| `RunConfig` | Used in Cell 6 to attach a guardrail at the run level instead of the agent level |
| `RunContextWrapper` | First parameter of every guardrail function |
| `Runner` | Runs agents. `Runner.run()` is awaited, `Runner.run_streamed()` is not |
| `TResponseInputItem` | The input item type in a guardrail signature |
| `input_guardrail`, `output_guardrail` | The two decorators |

In [13]:
import asyncio
import time

from pydantic import BaseModel, Field
from openai.types.responses import ResponseTextDeltaEvent
from openai.types.shared import Reasoning

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    ModelSettings,
    RunConfig,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
    output_guardrail,
)

print("Imports ready.")

Imports ready.


## Cell 5: How the SDK Actually Partitions Guardrails

No code to run in this cell. Read it, because everything after it makes more sense once you
have seen this.

At the top of the run loop, before the first turn of the agent starts, the SDK does this:

```python
all_input_guardrails = (
    starting_agent.input_guardrails + (run_config.input_guardrails or [])
    if current_turn == 0 and not is_resumed_state
    else []
)
sequential_guardrails = [g for g in all_input_guardrails if not g.run_in_parallel]
parallel_guardrails = [g for g in all_input_guardrails if g.run_in_parallel]
```

Five lines. They explain three things that were never spelled out when you first learned to
write an input guardrail.

### 1. Agent guardrails and RunConfig guardrails are concatenated

Look at the `+`. Guardrails attached to the agent and guardrails passed through `RunConfig`
are not alternatives and they do not override each other. They are joined into one list and
all of them run. Output guardrails follow exactly the same pattern:
`current_agent.output_guardrails + (run_config.output_guardrails or [])`.

Cell 6 proves this by attaching one guardrail to the agent and one to the `RunConfig`, then
counting the results.

### 2. `current_turn == 0` is the literal first-agent-only mechanism

You were told that input guardrails run only for the first agent in the chain. This is the
mechanism. On any turn after the first, `all_input_guardrails` evaluates to an empty list.
There is no per-agent bookkeeping and no special case for handoffs. The condition is simply
that the turn counter is still zero.

### 3. `run_in_parallel` is a partition key, not a timing hint

This is the one that changes how you think about the flag. `run_in_parallel` does not tell a
single guardrail to hurry up or hold back. The SDK splits your one list into **two separate
lists**, and those two lists run at two different points in the run loop.

| List | When it runs | Effect of a tripwire |
|---|---|---|
| `sequential_guardrails` (`run_in_parallel=False`) | Before the model task is created | The agent never starts |
| `parallel_guardrails` (`run_in_parallel=True`) | Inside `asyncio.gather()` alongside the model task | The agent may already be mid flight |

Once the sequential group passes, the SDK creates the model task and then gathers the
parallel group against it:

```python
parallel_results, turn_result = await asyncio.gather(
    run_input_guardrails(...),
    model_task,
)
```

That single line is the whole cost story. In the sequential group, the agent has not been
started yet. In the parallel group, it is already running. Cell 7 turns that difference into
a number.

## Cell 6: Reading Guardrail Results on a Successful Run

Up to now you have read guardrail decisions off the raised exception. That works, but it only
tells you about runs that were blocked. Every successful run carries the same information.

`RunResultBase` exposes two arrays:

| Attribute | Type |
|---|---|
| `result.input_guardrail_results` | `list[InputGuardrailResult]` |
| `result.output_guardrail_results` | `list[OutputGuardrailResult]` |

Straight from the SDK documentation: these arrays accumulate across the run, so they are
useful for logging decisions, storing extra guardrail metadata, or debugging why a run was
blocked. This is your audit path. It is where you go when someone asks what every check
decided on every request, not just the ones that failed.

Each entry gives you `r.guardrail.get_name()` for the name and `r.output` for the
`GuardrailFunctionOutput`, which carries both `tripwire_triggered` and whatever you put in
`output_info`.

This cell also demonstrates fact one from Cell 5. `topic_guardrail` is attached to the agent.
`input_length_guardrail` is passed through `RunConfig`. Neither replaces the other. Count the
results and you will see two.

The cell defines a `TopicCheckOutput` model and a `topic_checker` agent, wraps that agent in
`topic_guardrail`, adds a cheap non-model `input_length_guardrail`, defines a
`length_guardrail` output check, builds `guarded_agent` with the topic check on input and the
length check on output, and runs it with the extra `RunConfig` guardrail attached.

There is also `tool_input_guardrail_results` and `tool_output_guardrail_results` on the same
result object for per-tool checks. Those belong to a later section of the course.

In [14]:
class TopicCheckOutput(BaseModel):
    is_on_topic: bool
    reasoning: str


topic_checker = Agent(
    name="Topic Checker",
    instructions=(
        "Check if the user's message is about customer support topics. "
        "Return is_on_topic accordingly."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=TopicCheckOutput,
)


@input_guardrail
async def topic_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_checker, input, context=ctx.context)
    check: TopicCheckOutput = result.final_output
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_on_topic,
    )


@input_guardrail
async def input_length_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    text = input if isinstance(input, str) else str(input)
    return GuardrailFunctionOutput(
        output_info={"input_chars": len(text)},
        tripwire_triggered=len(text) > 500,
    )


class LengthCheckOutput(BaseModel):
    char_count: int
    within_limit: bool


@output_guardrail
async def length_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str,
) -> GuardrailFunctionOutput:
    count = len(str(output))
    return GuardrailFunctionOutput(
        output_info=LengthCheckOutput(
            char_count=count,
            within_limit=count <= 2000,
        ),
        tripwire_triggered=count > 2000,
    )


guarded_agent = Agent(
    name="Guarded Support Agent",
    instructions="You are a customer support agent. Be concise.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[topic_guardrail],
    output_guardrails=[length_guardrail],
)

result = await Runner.run(
    guarded_agent,
    "What's your return policy?",
    run_config=RunConfig(input_guardrails=[input_length_guardrail]),
)

print("Final output:", result.final_output[:100])

print("\nInput guardrail results:", len(result.input_guardrail_results))
for r in result.input_guardrail_results:
    print(f"  {r.guardrail.get_name()}: triggered={r.output.tripwire_triggered}")
    print(f"    info: {r.output.output_info}")

print("\nOutput guardrail results:", len(result.output_guardrail_results))
for r in result.output_guardrail_results:
    print(f"  {r.guardrail.get_name()}: triggered={r.output.tripwire_triggered}")
    print(f"    info: {r.output.output_info}")

Final output: Most items can be returned within 30 days of delivery, as long as they’re unused and in original pac

Input guardrail results: 2
  input_length_guardrail: triggered=False
    info: {'input_chars': 26}
  topic_guardrail: triggered=False
    info: is_on_topic=True reasoning='The user is asking about a return policy, which is a customer support topic.'

Output guardrail results: 1
  length_guardrail: triggered=False
    info: char_count=182 within_limit=True


## Cell 7: Measuring the Real Token Cost of Parallel vs Blocking

Cell 5 explained why the two execution modes cost different amounts. This cell measures it.

Both agents below carry a guardrail that always trips. The only difference between them is
one keyword:

| Agent | Guardrail | What the run loop does |
|---|---|---|
| `parallel_agent` | `run_in_parallel=True` | Starts the model task, then gathers the guardrail against it |
| `blocking_agent` | `run_in_parallel=False` | Runs the guardrail to completion first, and only then creates the model task |

Both guardrails use `asyncio.sleep(GUARDRAIL_LATENCY)` instead of calling a checker model.
That is deliberate. A real checker model takes an unpredictable amount of time, which would
make the comparison non deterministic. A fixed delay stands in for that round trip and
guarantees that in parallel mode the main agent's own call has time to finish before the
tripwire fires. That is exactly the situation you are trying to measure.

Usage is read off the raised exception. `e.run_data` is a `RunErrorDetails` object, and
`e.run_data.context_wrapper.usage` gives you the same `Usage` object you would get from a
successful run.

**One thing to be clear about.** These numbers are the *main agent's* wasted tokens only.
When a guardrail calls `Runner.run()` on a checker agent, that nested run gets its own
`RunContextWrapper` and its own usage counter. The guardrail's own cost is the same in both
modes, so it is not what the comparison is about.

If both modes report zero, your model finished slower than `GUARDRAIL_LATENCY`. Raise the
delay and run the cell again.

In [15]:
GUARDRAIL_LATENCY = 20.0  # Seconds. Stands in for a real checker model's round trip.


@input_guardrail(run_in_parallel=True)
async def always_block_parallel(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    await asyncio.sleep(GUARDRAIL_LATENCY)
    return GuardrailFunctionOutput(
        output_info={"mode": "parallel"},
        tripwire_triggered=True,
    )


@input_guardrail(run_in_parallel=False)
async def always_block_blocking(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    await asyncio.sleep(GUARDRAIL_LATENCY)
    return GuardrailFunctionOutput(
        output_info={"mode": "blocking"},
        tripwire_triggered=True,
    )


parallel_agent = Agent(
    name="Parallel Blocked Agent",
    instructions="You are a helpful assistant. Write a detailed, comprehensive answer.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[always_block_parallel],
)

blocking_agent = Agent(
    name="Blocking Blocked Agent",
    instructions="You are a helpful assistant. Write a detailed, comprehensive answer.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[always_block_blocking],
)

msg = "Explain the history of the internet in detail."

try:
    await Runner.run(parallel_agent, msg)
except InputGuardrailTripwireTriggered as e:
    usage = e.run_data.context_wrapper.usage
    print(
        f"Parallel mode: {usage.total_tokens} tokens wasted "
        f"across {usage.requests} model request(s)"
    )

try:
    await Runner.run(blocking_agent, msg)
except InputGuardrailTripwireTriggered as e:
    usage = e.run_data.context_wrapper.usage
    print(
        f"Blocking mode: {usage.total_tokens} tokens wasted "
        f"across {usage.requests} model request(s)"
    )

Parallel mode: 1839 tokens wasted across 1 model request(s)
Blocking mode: 0 tokens wasted across 0 model request(s)


## Cell 8: Sibling Cancellation When a Tripwire Fires

You now know how guardrails are partitioned across the run loop. This cell is about what
happens *inside* one of those groups when a check fails.

Guardrails within a group do not run one after another. The SDK wraps each one in
`asyncio.create_task()` and consumes them with `asyncio.as_completed()`, which yields them in
finishing order rather than declaration order. The moment one of them reports
`tripwire_triggered`, this runs:

```python
for t in guardrail_tasks:
    t.cancel()
await asyncio.gather(*guardrail_tasks, return_exceptions=True)
raise InputGuardrailTripwireTriggered(result)
```

Every sibling still in flight is cancelled, awaited so nothing leaks, and the exception is
raised. The run does not wait for the slow ones to finish. There would be no point. The run
is already stopping. The same code path exists in `run_output_guardrails`, so output groups
behave identically.

The cell defines three guardrails on one agent. `fast_tripwire` sleeps for a tenth of a
second and then trips. `slow_check_a` and `slow_check_b` each sleep for three seconds and
would pass. It then runs the agent and times the whole thing.

Watch the elapsed time, and watch which print statements actually appear.

In [16]:
@input_guardrail
async def fast_tripwire(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    await asyncio.sleep(0.1)
    print("  [fast_tripwire] completing, TRIPPING")
    return GuardrailFunctionOutput(
        output_info={"speed": "fast"},
        tripwire_triggered=True,
    )


@input_guardrail
async def slow_check_a(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    await asyncio.sleep(3.0)
    print("  [slow_check_a] completing, you will NOT see this")
    return GuardrailFunctionOutput(
        output_info={"speed": "slow"},
        tripwire_triggered=False,
    )


@input_guardrail
async def slow_check_b(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    await asyncio.sleep(3.0)
    print("  [slow_check_b] completing, you will NOT see this")
    return GuardrailFunctionOutput(
        output_info={"speed": "slow"},
        tripwire_triggered=False,
    )


race_agent = Agent(
    name="Race Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[fast_tripwire, slow_check_a, slow_check_b],
)

start = time.time()
try:
    await Runner.run(race_agent, "Hello there.")
except InputGuardrailTripwireTriggered as e:
    elapsed = time.time() - start
    print(f"\nTripped by: {e.guardrail_result.guardrail.get_name()}")
    print(f"Elapsed: {elapsed:.1f}s")
    print(
        "The two 3-second guardrails were cancelled mid flight. "
        "The run did not wait for them."
    )

  [fast_tripwire] completing, TRIPPING

Tripped by: fast_tripwire
Elapsed: 0.2s
The two 3-second guardrails were cancelled mid flight. The run did not wait for them.


## Cell 9: Streaming Guardrails, Checking Every N Characters

Output guardrails have a timing problem that no configuration flag can fix. They run on the
*final* output. For a response of a thousand tokens streaming to a user's screen, the user
has already read most of it before your check gets to look at anything.

This cell builds the workaround: check the partial text as it streams, and stop early if it
goes wrong.

Here is how it works. The cell defines a `ReadabilityOutput` model and a
`readability_checker` agent that judges whether text could be understood by a ten year old.
`check_readability` wraps that agent in a plain async function. `verbose_agent` is
deliberately instructed to write long, detailed responses so there is a stream worth
interrupting. Then the streaming loop runs.

Three details in the loop are worth naming.

| Detail | Why |
|---|---|
| `Runner.run_streamed(...)` is not awaited | It returns a `RunResultStreaming` immediately. You await the events, not the call |
| `asyncio.create_task(check_readability(...))` | The check runs off the streaming path, so tokens keep flowing while it works |
| `if ... and not guardrail_task` | Only one check runs at a time |

That last guard comes with a note from the SDK's own example: we do not run the guardrail
check if there is already a task running. An alternate implementation is to have N guardrails
running, or cancel the previous one.

Every iteration of the loop asks `guardrail_task.done()`. If the finished check says the text
is not readable, the loop breaks and the stream stops right there.

**This is a pattern you build, not an SDK feature.** The SDK's `output_guardrails` still run
only at the end. Nothing here changes that. What you are doing is adding your own check
alongside the stream because the built-in one arrives too late for this use case.

In [17]:
class ReadabilityOutput(BaseModel):
    reasoning: str = Field(
        description=(
            "Reasoning about whether the response could be understood "
            "by a ten year old."
        )
    )
    is_readable_by_ten_year_old: bool = Field(
        description="Whether the response is understandable by a ten year old."
    )


readability_checker = Agent(
    name="Readability Checker",
    instructions=(
        "You will be given a question and a response. Judge whether the "
        "response is simple enough to be understood by a ten year old."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=ReadabilityOutput,
)


async def check_readability(text: str) -> ReadabilityOutput:
    result = await Runner.run(readability_checker, text)
    return result.final_output_as(ReadabilityOutput)


verbose_agent = Agent(
    name="Verbose Assistant",
    instructions=(
        "You are a helpful assistant. You ALWAYS write long responses, "
        "making sure to be verbose and detailed."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="high",
    ),
)

question = "What is a black hole, and how does it behave?"
result = Runner.run_streamed(verbose_agent, question)

current_text = ""
next_check_len = 300
guardrail_task = None
tripped = False

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)
        current_text += event.data.delta

        if len(current_text) >= next_check_len and not guardrail_task:
            print("\n[Running readability check]\n", flush=True)
            guardrail_task = asyncio.create_task(check_readability(current_text))
            next_check_len += 300

    if guardrail_task and guardrail_task.done():
        check = guardrail_task.result()
        if not check.is_readable_by_ten_year_old:
            print("\n\n=== GUARDRAIL TRIGGERED MID STREAM ===")
            print(f"Reason: {check.reasoning}")
            tripped = True
            break
        guardrail_task = None

if not tripped:
    final_check = await check_readability(current_text)
    print("\n\n=== STREAM COMPLETED ===")
    print(f"Readable by a ten year old: {final_check.is_readable_by_ten_year_old}")
    print(f"Reason: {final_check.reasoning}")

A **black hole** is a region of space where gravity is so strong that **nothing can escape from inside it**, not even light. Because light cannot get out, black holes are invisible directly. We detect them by how they affect nearby matter, light, and the motion of stars.

## What creates a black hole
[Running readability check]

?
Most black holes form when a **very massive star** runs out of fuel and collapses under its own gravity. If the star is massive enough, the collapse continues until matter is squeezed into an extremely small region.

There are also:
- **Supermassive black holes** at the centers of galaxies, with masses millions to billions of times the Sun’s mass.
- **Stellar-mass black holes**, formed from collapsing stars.
- Possibly **intermediate-mass black holes**, which are harder to find.
- Hypothetical **primordial black holes**, which may have formed in the early universe.

## Main parts of a black hole
A black hole is usually described by a few key features:

- **Ev

## Cell 10: A Complete System, Input Plus Output Plus Handoffs

Everything in this section comes together here.

The cell builds two agents. `specialist` answers detailed policy questions and carries the
`length_guardrail` on its output. `front_door` is a triage agent that carries the
`topic_guardrail` on its input and hands off to `specialist`. Then it runs `front_door` and
prints which guardrails actually fired.

Notice what neither agent has. Neither one carries both kinds of guardrail. And yet the full
entry to exit path of the conversation is covered:

| Guardrail | Lives on | Fires because |
|---|---|---|
| `topic_guardrail` (input) | `front_door` | `front_door` is the first agent, so `current_turn == 0` |
| `length_guardrail` (output) | `specialist` | `specialist` ends up as the last agent after the handoff |

This is the design lesson of the whole section. Put input guardrails on your entry point. Put
output guardrails on whichever agents can produce a final answer. The run loop's scoping
rules do the routing for you.

In [18]:
specialist = Agent(
    name="Policy Specialist",
    instructions="You answer detailed policy questions. Be concise.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_guardrails=[length_guardrail],
)

front_door = Agent(
    name="Front Door",
    instructions=(
        "You are a triage agent. Hand off detailed policy questions "
        "to the Policy Specialist."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[topic_guardrail],
    handoffs=[specialist],
)

result = await Runner.run(
    front_door,
    "What is your full return and refund policy?",
)

print("Final output:", result.final_output[:150])
print("Last agent:", result.last_agent.name)
print(
    "Input guardrails that ran:",
    [r.guardrail.get_name() for r in result.input_guardrail_results],
)
print(
    "Output guardrails that ran:",
    [r.guardrail.get_name() for r in result.output_guardrail_results],
)

Final output: I don’t have a specific company’s return/refund policy unless you provide it. If you’re asking for a generic policy template, here’s a concise version
Last agent: Policy Specialist
Input guardrails that ran: ['topic_guardrail']
Output guardrails that ran: ['length_guardrail']


## Cell 11: What Belongs in a Guardrail vs Agent Instructions

No code here. This is the judgement call you will make every time you add a new rule to a
system, and getting it wrong in either direction is expensive.

| Put it in a guardrail | Put it in instructions |
|---|---|
| Hard policy that must never be bypassed | Tone, style and formatting preferences |
| Checks that need a separate, cheaper model | Behaviour the main model can self enforce reliably |
| Anything you need an audit record of | Soft guidance where occasional deviation is acceptable |
| Checks that should halt execution entirely | Guidance the model should weigh against other goals |

The distinction underneath the table is enforcement. Instructions are guidance the model
weighs against everything else in its context. A guardrail is code, and code either passes or
it does not. If a rule genuinely must not be negotiable, it does not belong in a prompt.

## Cell 12: Execution Mode Decision Guide

No code here either. Use this as your reference when you are deciding how a new check should
run.

| Mode | Latency | Wasted tokens on a trip | Use when |
|---|---|---|---|
| Input, `run_in_parallel=True` (default) | None added when the check passes | The main agent may have started, so those tokens are consumed | Trips are rare and latency matters |
| Input, `run_in_parallel=False` | Always adds the guardrail's duration | Zero, because the agent never starts | Trips are expensive or dangerous |
| Output guardrails | Always adds duration after the agent finishes | The full agent output has already been generated | Always, because there is no alternative |
| Streaming check every N characters (hand rolled) | None, it runs alongside the stream | Partial output has already been shown to the user | Long streamed responses where waiting for the end is too late |

One more mechanism worth knowing about. Every guardrail runs inside its own
`guardrail_span`, and the span records whether the tripwire fired. That means your guardrail
decisions are already visible in tracing without you writing any logging code. Tracing is the
subject of the next section.

## Cell 13: Section 5 Recap

That is the end of Section 5. Ten lectures, and here is the whole arc in one table.

| Lecture | What it covered |
|---|---|
| 5.1 | Orchestration patterns: LLM driven vs code driven |
| 5.2 | Handoffs: the full Handoff API, `on_handoff`, `input_type`, `is_enabled` |
| 5.3 | Handoff input filters and `HandoffInputData` |
| 5.4 | `as_tool()` vs handoffs: the decision framework |
| 5.5 | Triage agents with bidirectional handoffs |
| 5.6 | Deterministic pipelines and evaluator loops |
| 5.7 | Parallelisation with `asyncio.gather` |
| 5.8 | Input guardrails and `run_in_parallel` |
| 5.9 | Output guardrails and last agent only scope |
| 5.10 | Tripwire mechanics, execution modes and cost |

You started this section able to run one agent. You are ending it able to route work between
several agents, delegate to specialists, run branches in parallel, and put enforced checks on
both ends of the conversation.

Section 6 turns the lights on. Tracing, observability, and the capstone project.